# Vídeo: OpenCV, modelo preentrenado por frame y CNN + LSTM
Este notebook propone una progresión práctica para trabajar con vídeo: primero técnicas clásicas con OpenCV, después clasificación frame a frame con un modelo preentrenado y, por último, una red CNN + LSTM para introducir la dimensión temporal.

## Hoja de ruta
1. **OpenCV + detección/seguimiento**: detectar movimiento y seguir objetos en un vídeo.
2. **Modelo preentrenado por frame**: aplicar una CNN preentrenada a fotogramas individuales.
3. **CNN + LSTM**: tratar el vídeo como una secuencia de imágenes para capturar información temporal.

Idea clave: con imágenes estáticas solemos clasificar una sola entrada; con vídeo podemos clasificar secuencias, detectar objetos, seguirlos, reconocer acciones o localizar eventos.

In [ ]:
# Librerías
from pathlib import Path
import os

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display, Markdown

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
except Exception as e:
    print("TensorFlow/Keras no disponible:", e)

print("Directorio actual:", os.getcwd())

## Selección de vídeo de trabajo
Puedes usar un archivo existente, subirlo en Colab o reutilizar una grabación hecha con la cámara.

In [ ]:
VIDEO_PATH = Path("video.mp4")

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded)))
except Exception:
    pass

print(f"Vídeo seleccionado: {VIDEO_PATH}")
print(f"¿Existe?: {VIDEO_PATH.exists()}")

In [ ]:
# Vista previa rápida
if VIDEO_PATH.exists():
    display(Video(str(VIDEO_PATH), embed=True))
else:
    print("No se ha encontrado el vídeo. Puedes seguir leyendo el notebook o aportar un archivo más adelante.")

## 1. OpenCV: detección de movimiento y seguimiento
Aquí no hay aprendizaje profundo. Trabajamos con operaciones clásicas sobre frames: diferencias, umbrales, contornos y algoritmos de tracking.

In [ ]:
# Detección de movimiento con sustracción de fondo (ejecutar localmente)
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=25, detectShadows=True)

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    mask = bg_subtractor.apply(frame)
    _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        if cv2.contourArea(contour) < 1200:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow("Movimiento", frame)
    cv2.imshow("Mascara", mask)

    if cv2.waitKey(30) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# Seguimiento simple con CSRT después de seleccionar una región (ejecutar localmente)
cap = cv2.VideoCapture(str(VIDEO_PATH))
ret, frame = cap.read()

if not ret or frame is None:
    cap.release()
    raise ValueError("No se pudo leer el primer frame para inicializar el tracker.")

tracker = cv2.TrackerCSRT_create()
bbox = cv2.selectROI("Selecciona objeto", frame, fromCenter=False, showCrosshair=True)
cv2.destroyWindow("Selecciona objeto")
tracker.init(frame, bbox)

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    ok, bbox = tracker.update(frame)
    if ok:
        x, y, w, h = [int(v) for v in bbox]
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
        cv2.putText(frame, "Tracking", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
    else:
        cv2.putText(frame, "Objeto perdido", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    cv2.imshow("Seguimiento CSRT", frame)
    if cv2.waitKey(30) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

### Qué aporta esta fase
- Detectar movimiento sin entrenar ningún modelo.
- Seguir objetos a lo largo del vídeo.
- Entender que el vídeo no siempre requiere deep learning para tareas básicas.

## 2. Modelo preentrenado por frame
Aquí aplicamos una CNN preentrenada a fotogramas individuales. Es una buena transición desde imagen fija: tratamos el vídeo como una colección de frames.

Limitación importante: un modelo entrenado sobre imágenes estáticas puede reconocer objetos presentes en cada frame, pero no entiende bien acciones o dinámicas temporales.

In [ ]:
# Extraer algunos frames de ejemplo
def sample_frames(video_path, num_frames=6):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"No se pudo abrir el vídeo: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        total = num_frames
    indices = np.linspace(0, max(total - 1, 0), num_frames, dtype=int)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret and frame is not None:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    cap.release()
    return frames

if VIDEO_PATH.exists():
    frames = sample_frames(VIDEO_PATH, num_frames=6)
    plt.figure(figsize=(12, 5))
    for i, frame in enumerate(frames):
        plt.subplot(2, 3, i + 1)
        plt.imshow(frame)
        plt.axis("off")
        plt.title(f"Frame {i + 1}")
    plt.tight_layout()
    plt.show()
else:
    print("Aporta un vídeo para extraer frames de ejemplo.")

In [ ]:
# Clasificación frame a frame con MobileNetV2 preentrenada en ImageNet
if not VIDEO_PATH.exists():
    print("No hay vídeo cargado.")
else:
    model = MobileNetV2(weights="imagenet")
    frames = sample_frames(VIDEO_PATH, num_frames=4)

    batch = []
    for frame in frames:
        resized = cv2.resize(frame, (224, 224)).astype("float32")
        batch.append(preprocess_input(resized))
    batch = np.array(batch)

    preds = model.predict(batch, verbose=0)
    decoded = decode_predictions(preds, top=3)

    for i, top in enumerate(decoded):
        print(f"\nFrame {i + 1}")
        for _, label, score in top:
            print(f"  {label}: {score:.3f}")

### Qué aporta esta fase
- Reutilizar una CNN entrenada para imágenes.
- Analizar un vídeo como secuencia de fotogramas.
- Entender la diferencia entre reconocer objetos en un frame y reconocer acciones en una secuencia.

## 3. CNN + LSTM: introducir la dimensión temporal
Cuando queremos clasificar acciones o eventos en vídeo, necesitamos modelar cómo cambian los frames a lo largo del tiempo. Una estrategia clásica es:

1. una **CNN** extrae características visuales de cada frame,
2. una **LSTM** procesa la secuencia de características,
3. una capa final predice la acción o clase del vídeo.

Este enfoque funciona bien como introducción didáctica antes de pasar a 3D CNN o transformers de vídeo.

In [ ]:
# Preparación de secuencias de frames desde vídeos organizados por clase
# Estructura esperada:
# dataset_video/
#   caminar/
#   correr/
#   saltar/

SEQ_LEN = 16
IMG_SIZE = (128, 128)
DATASET_DIR = Path("vision/data/video_dataset_demo")

def extract_sequence(video_path, seq_len=SEQ_LEN, img_size=IMG_SIZE):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        total = seq_len
    indices = np.linspace(0, max(total - 1, 0), seq_len, dtype=int)

    sequence = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret or frame is None:
            cap.release()
            return None
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, img_size).astype("float32") / 255.0
        sequence.append(frame)

    cap.release()
    return np.array(sequence)

def load_video_dataset(dataset_dir):
    X, y = [], []
    class_names = sorted([p.name for p in dataset_dir.iterdir() if p.is_dir()])
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    for class_name in class_names:
        class_dir = dataset_dir / class_name
        for video_path in class_dir.glob("*.mp4"):
            seq = extract_sequence(video_path)
            if seq is not None:
                X.append(seq)
                y.append(class_to_idx[class_name])

    if not X:
        return None, None, class_names

    return np.array(X), np.array(y), class_names

if DATASET_DIR.exists():
    X, y, class_names = load_video_dataset(DATASET_DIR)
    if X is not None:
        print("X:", X.shape)
        print("y:", y.shape)
        print("Clases:", class_names)
    else:
        print("No se han podido cargar secuencias válidas.")
else:
    print("Crea primero un pequeño dataset de vídeos por carpetas si quieres entrenar esta parte.")

In [ ]:
# Modelo CNN + LSTM sencillo
cnn_lstm = models.Sequential([
    layers.TimeDistributed(layers.Conv2D(32, (3, 3), activation="relu"), input_shape=(SEQ_LEN, IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.TimeDistributed(layers.MaxPooling2D((2, 2))),
    layers.TimeDistributed(layers.Conv2D(64, (3, 3), activation="relu")),
    layers.TimeDistributed(layers.MaxPooling2D((2, 2))),
    layers.TimeDistributed(layers.Flatten()),
    layers.LSTM(128),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(3, activation="softmax")
])

cnn_lstm.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn_lstm.summary()

In [ ]:
# Entrenamiento de ejemplo
# Sustituye el número de clases si tu dataset no tiene exactamente 3.

if "X" in globals() and X is not None and len(np.unique(y)) >= 2:
    num_classes = len(np.unique(y))
    cnn_lstm.layers[-1] = layers.Dense(num_classes, activation="softmax")
    cnn_lstm.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    history = cnn_lstm.fit(X, y, validation_split=0.2, epochs=5, batch_size=4)
else:
    print("Esta celda es un esqueleto de entrenamiento. Necesitas un pequeño dataset de vídeos etiquetados.")

## Variante más potente: CNN preentrenada + LSTM
En vez de entrenar la parte convolucional desde cero, también puedes usar una CNN preentrenada por frame como extractor de características y pasar esas características a una LSTM. Es una combinación muy habitual cuando el dataset de vídeo es pequeño.

In [ ]:
# Esqueleto: MobileNetV2 como extractor por frame + LSTM
base_cnn = MobileNetV2(weights="imagenet", include_top=False, pooling="avg", input_shape=(224, 224, 3))
base_cnn.trainable = False

feature_model = models.Sequential([
    layers.TimeDistributed(base_cnn, input_shape=(SEQ_LEN, 224, 224, 3)),
    layers.LSTM(128),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(3, activation="softmax")
])

feature_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
feature_model.summary()

## Conclusión didáctica
- **OpenCV** resuelve tareas sencillas de movimiento y seguimiento sin entrenar modelos.
- **Modelo preentrenado por frame** reutiliza lo aprendido en imagen estática, pero todavía no modela bien el tiempo.
- **CNN + LSTM** introduce la secuencia como objeto de aprendizaje y sirve como puente hacia técnicas más avanzadas de vídeo.

En problemas reales de vídeo también son frecuentes YOLO + tracking, 3D CNN, Video Transformers y modelos multimodales.

In [ ]:
Markdown('''
**Actividad sugerida**

1. Usa OpenCV para detectar y seguir un objeto en un vídeo corto.
2. Extrae algunos frames y clasifícalos con un modelo preentrenado.
3. Diseña una mini colección de vídeos etiquetados por acción y prueba una CNN + LSTM.
4. Compara entrenar la CNN desde cero frente a reutilizar un extractor preentrenado.
''')